# Grouping and summarizing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/12-grouping-and-summarizing.ipynb)

A table of 2,000 sales is too long to read. A manager asks short questions instead: Which store sells the most?
Which product category brings in the most money? Which month was the best? In this chapter, we answer such
questions with one line of pandas each: we put the rows into groups and compute one number for each group. Then we
draw our first charts.

**In this chapter, you will learn to:**

- count how often each value of a column appears, with `value_counts()`;
- compute a total, an average or a count for each group, with `groupby()`;
- group by two columns, compute several summaries at once with `agg()`, and turn a result back into a DataFrame;
- draw a bar chart and a line chart of a summary, with matplotlib.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

Besides pandas, the first cell imports **matplotlib**, the most widely used Python package for charts.
`matplotlib.pyplot` is the part of it that draws charts, and `plt` is its usual short name. We use it at the end
of the chapter.

## The data

We use the sales of the chain's eight stores again (the chapter *The datasets* describes the file). Each chapter
runs on its own, so we first add the two columns that we made in the chapter *New columns and sorting*: the
revenue of each sale, and its month.

In [ ]:
sales = pd.read_csv(DATA_URL + "sales.csv")
sales["revenue"] = (sales["units"] * sales["price"] * (1 - sales["discount"])).round(2)
sales["month"] = sales["date"].str[:7]
sales.head()

Each row is one sale: the day, the store and its region, the product category, the number of units, the price of
one unit and the discount. `revenue` is the money that the store received, in dollars, and `month` holds the first
seven characters of the date, such as `2025-01`.

## Counting values with value_counts()

The simplest summary is a count. How many sales did each store make?

In [ ]:
sales["store"].value_counts()

`value_counts()` counts how often each value of a column appears. The result is a Series: its index holds the
values (here, the stores), and its values are the counts, from the largest to the smallest. Store S1 made the most
sales, 287, and store S8 the fewest, 226. Together, the eight counts add up to the 2,000 rows. (`count` under the
result is the name of the Series.)

Often we want shares instead of counts: what fraction of the sales is in each category? The option
`normalize=True` divides each count by the total:

In [ ]:
sales["category"].value_counts(normalize=True)

The shares add up to 1. About 40% of the sales (0.397) are grocery sales, and only about 14% (0.1435) are
electronics sales. This time the name of the Series is `proportion`, another word for a share.

> **Try it:** count the sales of each `region`. Which region has the most sales?

## Grouping: one number for each group

`value_counts()` can only count. A manager also wants totals: how much **revenue** did each store make? For that,
we must put the rows of each store together, and add up their revenue. Let's start with a table so small that we
can check the result by hand:

In [ ]:
small = pd.DataFrame({
    "store": ["S1", "S2", "S1", "S2", "S1"],
    "revenue": [100, 40, 50, 60, 30],
})
small.groupby("store")["revenue"].sum()

Read the last line from left to right:

1. `small.groupby("store")` puts the rows into **groups**, one group for each store: the rows of S1 (revenue 100,
   50 and 30) and the rows of S2 (40 and 60).
2. `["revenue"]` chooses the column to summarize.
3. `.sum()` adds up the revenue in each group: 180 for S1 and 100 for S2.

The result is a Series with one value for each group, and the groups as its index, in alphabetical order. This
pattern, *split* the rows into groups, *apply* a summary to each group, and *combine* the results, is often called
**split-apply-combine**.

Now the same line on the real data:

In [ ]:
sales.groupby("store")["revenue"].sum()

2,000 rows became 8 numbers. Store S5 made the most revenue, about $47,128, and store S8 the least, about $32,675.
Notice that S1, which made the most sales, does not have the most revenue: the number of sales is not the whole
story.

## Other summaries: mean, count and size

Instead of `sum()`, we can use the other summaries that we know from the chapter *Comparisons and Boolean masks*,
such as `mean()`, `min()` and `max()`. Here is the average revenue of one sale in each store, rounded to cents:

In [ ]:
sales.groupby("store")["revenue"].mean().round(2)

Store S5 has the largest average sale, $180.57, and store S6 the smallest, $140.61. So S5 is first in total revenue
because its sales are bigger: it made fewer sales than S1 and S2.

To count the rows of each group, we use `size()`. It needs no column:

In [ ]:
sales.groupby("store").size()

These are the same numbers as `value_counts()` gave above, but in the order of the stores. `count()`, as in
`sales.groupby("store")["revenue"].count()`, is almost the same: it counts the values of the chosen column in each
group, but it skips missing values. `sales.csv` has no missing values, so here `count()` and `size()` agree. The
next chapter is about data with missing values.

## Choose the number columns first

What happens if we forget to choose a column?

In [ ]:
sales.groupby("store").mean()

Without a column, `groupby` tries to average **every** column, and some columns hold text. The last line of the
error says so. In Colab (pandas 2) it is `TypeError: agg function failed [how->mean,dtype->object]`: the `mean`
failed on a column of the dtype `object`, which is text. In pandas 3 it reads `TypeError: dtype 'str' does not
support operation 'mean'`. The average of a date or of a store name makes no sense.

The fix is to choose the number columns before the summary. One column takes single brackets, as above. Several
columns take double brackets, and give a DataFrame:

In [ ]:
sales.groupby("store")[["units", "revenue"]].sum()

Now each store has two totals: the units it sold and its revenue. Store S1 sold the most units, 1,346.

> **Common mistake:** `sales.groupby("store").sum()` gives **no** error, but its result is useless. pandas "adds"
> text by gluing it together, so the `date` column becomes one very long piece of text for each store. Always
> choose the columns before `sum()` or `mean()`. (Another way is `numeric_only=True`, as in
> `sales.groupby("store").sum(numeric_only=True)`, which keeps only the number columns.)

## Sorting a grouped result

A grouped result comes in the order of the groups. To see the best stores first, we sort it by its values. A
Series has `sort_values()` too, and needs no column name:

In [ ]:
store_revenue = sales.groupby("store")["revenue"].sum()
store_revenue.sort_values(ascending=False)

We kept the result in a variable, `store_revenue`, because we use it again for a chart. Sorted, the ranking is
clear: S5, then S2 and S1, and S6 and S8 at the bottom.

> **Try it:** add `.head(3)` at the end of the second line to see only the top 3 stores.

## Grouping by two columns

Is the best category the same in every region? To answer, we need one group for each **combination** of a region
and a category. We pass a list of two columns to `groupby`:

In [ ]:
sales.groupby(["region", "category"])["revenue"].sum()

There are 4 regions and 4 categories, so there are 16 groups. The index now has two levels: the region, and the
category inside it. pandas writes each region only once, on the first of its four rows.

In every region, Electronics brings in the most revenue and Grocery the least. That is a surprise only at first:
we saw that about 40% of the sales are grocery sales. Grocery sales are many but small, and electronics sales are
few but big.

## Several summaries at once with agg()

To see both sides at once (how many sales, how big, and the total), we can ask for several summaries in one step.
`agg()` takes a list of summary names, as text:

In [ ]:
sales.groupby("category")["revenue"].agg(["sum", "mean", "count"]).round(2)

The result is a DataFrame with one column for each summary. Electronics has the fewest sales (287), but by far the
largest average sale ($452.58) and the largest total ($129,891.66). Grocery has the most sales (794) and the
smallest average ($56.42).

## Back to a DataFrame with reset_index()

A grouped result has the groups as its index. Sometimes we want an ordinary table instead, with the groups in a
column, for example to filter it as in the chapter *Filtering rows*. `reset_index()` moves the index into a
column. Here we use it, and then keep the stores that made more than $40,000:

In [ ]:
by_store = sales.groupby("store")["revenue"].sum().reset_index()
by_store[by_store["revenue"] > 40000]

`by_store` is a DataFrame with two columns, `store` and `revenue`, and the index 0 to 7, so a filter works on
it as usual. Five of the eight stores made more than $40,000: S1, S2, S3, S5 and S7. (The
index of the result skips 3, 5 and 7: those rows, S4, S6 and S8, were filtered out.)

> **Try it:** add `.reset_index()` to the grouping by region and category above. How many rows and columns does
> the result have?

## A bar chart

A chart shows a summary at a glance. A Series has a `plot()` method, which draws it with matplotlib. Here are the
stores' revenues, sorted, as a **bar chart**:

In [ ]:
store_revenue.sort_values(ascending=False).plot(kind="bar")
plt.title("Revenue by store")
plt.xlabel("Store")
plt.ylabel("Revenue ($)")
plt.show()

Line by line:

- `.plot(kind="bar")` draws one bar for each value of the Series. The index (the stores) gives the labels under
  the bars.
- `plt.title`, `plt.xlabel` and `plt.ylabel` add a title, and a label to each axis: the horizontal axis (x) and
  the vertical axis (y).
- `plt.show()` shows the chart.

Because we sorted first, the bars go from the tallest to the shortest, and they are easy to compare: S5 is ahead,
four stores follow close together, and S4, S6 and S8 are clearly behind.

> **In Thonny:** `plt.show()` opens the chart in a new window, and the program waits until you close the window.
> Then it runs the rest of the script.

> **Note:** end every chart cell with `plt.show()`. Without it, Thonny may show no chart at all, and Colab shows
> an extra line of text above the chart: the value of the cell's last line.

> **Try it:** change `kind="bar"` to `kind="barh"` and run the cell again. What changes?

## A line chart

When the groups follow each other in time, such as months, a **line chart** shows how the summary changes:

In [ ]:
monthly = sales.groupby("month")["revenue"].sum()
monthly.plot(kind="line", marker="o")
plt.title("Revenue by month")
plt.xlabel("Month")
plt.ylabel("Revenue ($)")
plt.show()

The months are text such as `2025-01`, and `groupby` puts them in alphabetical order, which here is also the
order in time. `marker="o"` draws a dot for each month, and the line joins the dots. The monthly revenue moves
between about $22,000 and $30,500: it is highest in March and lowest in November. (Under the axis, matplotlib
writes only every second month, to leave room.)

Look at the vertical axis: it starts at about 22,000, not at 0. matplotlib does this for a line chart, so that the
changes are easy to see, but it also makes them look bigger than they are. The best month brought in less than
40% more than the worst.

## Summary

- `df["col"].value_counts()` counts how often each value appears, the largest count first. With
  `normalize=True`, it gives shares.
- `df.groupby("key")["col"].sum()` puts the rows into groups by the column `key`, and computes one value for each
  group. `mean()`, `count()`, `min()` and `max()` work the same way. `df.groupby("key").size()` counts the rows of
  each group.
- Choose the number columns (`["col"]` or `[["a", "b"]]`) before `sum()` or `mean()`.
- A grouped result is a Series: sort it with `sort_values(ascending=False)`, and take the top with `head()`.
- `df.groupby(["a", "b"])` makes one group for each combination of two columns.
- `.agg(["sum", "mean", "count"])` computes several summaries at once, and gives a DataFrame.
- `reset_index()` turns the groups back into a column, so the result is an ordinary DataFrame.
- A chart: `series.plot(kind="bar")` (or `kind="line"`), then `plt.title(...)`, `plt.xlabel(...)`,
  `plt.ylabel(...)`, and always `plt.show()` at the end.

## Further reading

- The pandas tutorials "How to calculate summary statistics"
  (<https://pandas.pydata.org/docs/getting_started/intro_tutorials/06_calculate_statistics.html>) and "How do I
  create plots in pandas?" (<https://pandas.pydata.org/docs/getting_started/intro_tutorials/04_plotting.html>).

## Practice questions

### Question 1 (Easy)

What share of the sales had no discount (a `discount` of `0.0`)?

<details class="yr-answer">
<summary>Answer</summary>

`value_counts(normalize=True)` gives the share of each discount. About half of the sales (0.509) had no discount.

```python
sales["discount"].value_counts(normalize=True)
```

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

How many units were sold in each category? Which category sold the most units?

<details class="yr-answer">
<summary>Answer</summary>

Group by `category`, choose `units`, and add up. Grocery sold by far the most units: 5,577.

```python
sales.groupby("category")["units"].sum()
```

</details>

In [ ]:
# Write your answer here

### Question 3 (Medium)

What is the average price of one unit in each category? Round to cents, and show the most expensive category
first.

<details class="yr-answer">
<summary>Answer</summary>

Electronics is the most expensive category, with an average price of $315.46, and Grocery the cheapest, with
$8.43.

```python
sales.groupby("category")["price"].mean().round(2).sort_values(ascending=False)
```

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

Make a DataFrame `by_region` with two columns, `region` and `revenue` (the total revenue of the region), sorted
from the largest revenue to the smallest.

<details class="yr-answer">
<summary>Answer</summary>

Group and sum, then `reset_index()` gives a DataFrame, which we sort by its `revenue` column. North comes first,
with $85,775.35, and West last.

```python
by_region = sales.groupby("region")["revenue"].sum().reset_index()
by_region = by_region.sort_values("revenue", ascending=False)
by_region
```

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

Run `sales.groupby(["region", "store"]).size()`. Why does the result have only 8 rows, and not 4 × 8 = 32?

<details class="yr-answer">
<summary>Answer</summary>

`groupby` makes a group only for the combinations that appear in the data. Each store is in one region (S1 and S2
are in the North, and so on), so there are only 8 combinations of a region and a store.

```python
sales.groupby(["region", "store"]).size()
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Hard)

Do bigger discounts come with more units per sale? Compute the average number of units of a sale for each
discount level, and say what you see.

<details class="yr-answer">
<summary>Answer</summary>

Group by `discount` and average `units`. The averages are all between 4 and 5 units, and the biggest discount
(0.2) has the smallest average, 4.05. In this data, bigger discounts do not come with more units per sale.

```python
sales.groupby("discount")["units"].mean().round(2)
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Hard)

Draw a bar chart of the total revenue of each category, from the largest to the smallest, with a title and labels
on both axes.

<details class="yr-answer">
<summary>Answer</summary>

Group, sum, sort, and plot. Electronics has the tallest bar and Grocery the shortest.

```python
category_revenue = sales.groupby("category")["revenue"].sum()
category_revenue.sort_values(ascending=False).plot(kind="bar")
plt.title("Revenue by category")
plt.xlabel("Category")
plt.ylabel("Revenue ($)")
plt.show()
```

</details>

In [ ]:
# Write your answer here

### Question 8 (Challenge)

In each region, what share of the revenue comes from Electronics? In which region is that share the largest?

<details class="yr-answer">
<summary>Answer</summary>

Compute two Series with the regions as their index: the total revenue of each region, and the electronics revenue
of each region (filter the rows first). Dividing one by the other divides region by region. The share is largest
in the West: 0.441, or about 44%.

```python
region_revenue = sales.groupby("region")["revenue"].sum()
electronics = sales[sales["category"] == "Electronics"]
electronics_revenue = electronics.groupby("region")["revenue"].sum()
(electronics_revenue / region_revenue).round(3)
```

</details>

In [ ]:
# Write your answer here